# EDA — Jugadores, 5 grandes ligas (Understat), 5 temporadas

Exploración a nivel de **jugador** de las 5 grandes ligas, temporadas **2021-22 a 2025-26**
(13.980 jugador-temporada; 8.017 tras el filtro de minutos). Data de Understat, que sí trae xG/xA
—a diferencia de fbref, que dejó de publicarlo.

Todos los gráficos llevan **selector de temporada** y buscador de jugador. Los ejes están fijados
sobre las 5 temporadas: cambiar de temporada mueve los puntos, no la escala.


In [1]:
import sys

from functools import partial

import pandas as pd
import plotly.graph_objects as go

sys.path.insert(0, ".")
import insights as ins
from viz_theme import (
    LEAGUE_ORDER, FUENTE_UNDERSTAT, apply_theme, apply_plotly_theme, league_color,
    render_with_sidebar, render_select_chart, league_box_season_html,
)

apply_theme()
apply_plotly_theme()

# El crédito de la fuente que va debajo de cada gráfico se fija una vez
# acá, igual que en `web/charts/` — así el notebook y el sitio acreditan
# lo mismo y un gráfico nuevo lo hereda en vez de quedarse sin fuente.
render_with_sidebar = partial(render_with_sidebar, fuente=FUENTE_UNDERSTAT)
render_select_chart = partial(render_select_chart, fuente=FUENTE_UNDERSTAT)

## Carga y limpieza

Las 5 temporadas ya consolidadas por `consolidate_data.py` (`data/processed/players_all_seasons.csv`), que además de apilar las 25 liga-temporada ya decodifica las entidades HTML de los nombres (`M&#039;Bala Nzola`), que es un defecto del origen.

Lo que **no** hace el consolidado, a propósito, es tocar `team`: los jugadores transferidos a mitad de temporada traen los dos clubes separados por coma (`"Angers,Rennes"`) y quedarse con uno es decisión de análisis, no de consolidación. Acá se toma el último.

⚠️ **Cuidado con los nombres**: esta tabla no trae `player_id`, solo el nombre, y hay pares de jugadores homónimos (2 en La Liga). Para un scatter de una temporada da igual, pero **no se pueden armar series temporales por jugador cruzando por nombre** sin arrastrar el id desde los datos de tiros.

In [2]:
df = pd.read_csv("../data/processed/players_all_seasons.csv")
df["team"] = df["team"].str.split(",").str[-1].str.strip()
df["liga"] = pd.Categorical(df["liga"], categories=LEAGUE_ORDER, ordered=True)

SEASONS = sorted(df["temporada"].unique())
SEASON = SEASONS[-1]  # la que se muestra al abrir cada gráfico

print(f"{len(df)} jugador-temporada · {len(SEASONS)} temporadas: {SEASONS}")
df.head()

13980 jugador-temporada · 5 temporadas: ['2021-22', '2022-23', '2023-24', '2024-25', '2025-26']


,temporada,liga,number,player,team,apps,min,goals,a,xG,...,key_passes,np_goals,np_xg,yellow_cards,red_cards,xg_chain,xg_buildup,born,sub21,posicion
0,2021-22,Bundesliga,1,Christian Günter,Freiburg,34,3060,2,9,0.930578,...,69,2,0.930578,1,0,9.451996,5.646296,1993.0,False,Defensa
1,2021-22,Bundesliga,2,Rafal Gikiewicz,Augsburg,34,3060,0,0,0.000000,...,2,0,0.000000,0,0,1.895747,1.895747,1987.0,False,Portero
2,2021-22,Bundesliga,3,Maximilian Arnold,Wolfsburg,34,3004,4,2,1.370162,...,58,4,1.370162,4,0,10.245887,8.710278,1994.0,False,Medio
3,2021-22,Bundesliga,4,Yann Sommer,Borussia M.Gladbach,33,2970,0,1,0.000000,...,1,0,0.000000,2,0,4.352445,3.952069,1988.0,False,Portero
4,2021-22,Bundesliga,5,Oliver Baumann,Hoffenheim,33,2970,0,0,0.000000,...,0,0,0.000000,1,0,7.588192,7.588192,1990.0,False,Portero


### Filtro de minutos mínimos

`xG90`/`xA90` son tasas: con pocos minutos jugados son ruido puro (un
suplente con 200 minutos y una racha de suerte puede tener una tasa
disparatada). Nos quedamos con jugadores con **≥900 minutos** (~10 partidos
completos) para todo lo que sigue — scatters y boxplots.

In [3]:
MIN_MINUTES = 500
BASE_SIZE = 8  # tamaño del punto, se le pasa al buscador para que resaltar y
               # volver no cambie el tamaño respecto del estado inicial

before = len(df)
df = df[df["min"] >= MIN_MINUTES].reset_index(drop=True)
print(f"filtro >= {MIN_MINUTES} min: {before} -> {len(df)} jugadores")


def season_data():
    """`{temporada: [(liga, sub_df), ...]}` — lo que espera el selector."""
    return {s: [(liga, df[(df["temporada"] == s) & (df["liga"] == liga)])
                for liga in LEAGUE_ORDER]
            for s in SEASONS}


def eje(serie, frac=0.06):
    """Rango de eje con aire sobre TODAS las temporadas: si cada temporada
    reescalara sus ejes, comparar entre temporadas sería engañoso."""
    lo, hi = serie.min(), serie.max()
    pad = (hi - lo) * frac
    return [lo - pad, hi + pad]


# Las cajas de lectura de cada gráfico salen de `insights.py`, que calcula las
# frases desde estos mismos datos (ver el docstring de ese módulo para por qué
# no están escritas a mano).
def por_liga(generador):
    """Una entrada por cada combinación temporada x liga, que es el estado del
    que dependen los gráficos con filtro de liga."""
    return {f'{s}|{lg}': dict(zip(('fija', 'salta'), generador(df, s, lg)))
            for s in SEASONS
            for lg in [ins.LIGA_TODAS] + list(LEAGUE_ORDER)}


def por_temporada(generador, *extra):
    return {s: dict(zip(('fija', 'salta'), generador(df, s, *extra)))
            for s in SEASONS}


temporadas = season_data()
df.groupby(["temporada", "liga"], observed=True).size().unstack()

filtro >= 500 min: 13980 -> 9693 jugadores


liga,Bundesliga,Serie A,Ligue 1,La Liga,Premier League
temporada,,,,,
2021-22,338,420,406,418,392
2022-23,342,422,402,410,402
2023-24,341,412,347,409,393
2024-25,340,414,356,419,383
2025-26,343,415,362,415,392


## 1. Producción vs. expected

Tres scatters interactivos, un punto por jugador — pasa el mouse para ver
el detalle, busca un jugador o filtra por liga en la barra lateral.

### Goles vs. xG

Quién sobre o bajo-rendimió su expected goals. La línea punteada es
`goles = xG`: por encima, el jugador metió más de lo que sus ocasiones
merecían (mejor definición o algo de suerte); por debajo, le faltó pegada
o tuvo mala fortuna.

In [4]:
scatter_data = temporadas[SEASON]

fig = go.Figure()
for liga, sub in scatter_data:
    fig.add_trace(go.Scatter(
        x=sub["xG"], y=sub["goals"], mode="markers", name=liga,
        marker=dict(color=league_color(liga), size=BASE_SIZE, opacity=0.75,
                    line=dict(width=0.5, color="white")),
        customdata=sub[["player", "team"]],
        hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                       "xG: %{x:.1f}<br>Goles: %{y:.0f}<extra></extra>",
    ))

mx = max(df["goals"].max(), df["xG"].max()) * 1.05
# La línea y=x va DESPUÉS de las trazas de liga: el helper asume ese orden y si
# se agrega antes los índices quedan desalineados (el filtro esconde la liga
# equivocada). Es un bug real que ya pasó en este notebook.
fig.add_trace(go.Scatter(x=[0, mx], y=[0, mx], mode="lines",
                          line=dict(color="#c3c2b7", dash="dash", width=1.4),
                          hoverinfo="skip", showlegend=False))

sub_g = (f"¿Quién sobre/bajo-rendimió su expected goals? Jugadores con >={MIN_MINUTES} min · {{temporada}}")
fig.update_layout(
    title=dict(text="Goles vs. xG", subtitle=dict(text=sub_g.format(temporada=SEASON))),
    xaxis_title="xG", yaxis_title="Goles",
    xaxis=dict(range=eje(df["xG"])), yaxis=dict(range=eje(df["goals"])),
)
render_with_sidebar(fig, scatter_data, "xG", "goals", extra_traces=1, name_col="player",
                    search_label="jugador", width=760, height=580, base_size=BASE_SIZE,
                    season_data=temporadas, custom_cols=["player", "team"],
                    subtitle_template=sub_g,
                    insights={'que_mirar': ins.goles_xg_que_mirar(),
                              'por_que': ins.goles_xg_por_que(df),
                              'dinamico': por_liga(ins.goles_xg)})

### Asistencias vs. xA

Mismo concepto para creación de juego: quién generó más o menos asistencias
de las que sus pases de gol merecían.

In [5]:
scatter_data = temporadas[SEASON]

fig = go.Figure()
for liga, sub in scatter_data:
    fig.add_trace(go.Scatter(
        x=sub["xA"], y=sub["a"], mode="markers", name=liga,
        marker=dict(color=league_color(liga), size=BASE_SIZE, opacity=0.75,
                    line=dict(width=0.5, color="white")),
        customdata=sub[["player", "team"]],
        hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                       "xA: %{x:.1f}<br>Asistencias: %{y:.0f}<extra></extra>",
    ))

mx = max(df["a"].max(), df["xA"].max()) * 1.05
# La línea y=x va DESPUÉS de las trazas de liga: el helper asume ese orden y si
# se agrega antes los índices quedan desalineados (el filtro esconde la liga
# equivocada). Es un bug real que ya pasó en este notebook.
fig.add_trace(go.Scatter(x=[0, mx], y=[0, mx], mode="lines",
                          line=dict(color="#c3c2b7", dash="dash", width=1.4),
                          hoverinfo="skip", showlegend=False))

sub_a = (f"¿Quién sobre/bajo-rendimió su expected assists? Jugadores con >={MIN_MINUTES} min · {{temporada}}")
fig.update_layout(
    title=dict(text="Asistencias vs. xA", subtitle=dict(text=sub_a.format(temporada=SEASON))),
    xaxis_title="xA", yaxis_title="Asistencias",
    xaxis=dict(range=eje(df["xA"])), yaxis=dict(range=eje(df["a"])),
)
render_with_sidebar(fig, scatter_data, "xA", "a", extra_traces=1, name_col="player",
                    search_label="jugador", width=760, height=580, base_size=BASE_SIZE,
                    season_data=temporadas, custom_cols=["player", "team"],
                    subtitle_template=sub_a,
                    insights={'que_mirar': ins.asist_xa_que_mirar(),
                              'por_que': ins.asist_xa_por_que(df),
                              'dinamico': por_liga(ins.asist_xa)})

### Perfil ofensivo: xG90 vs. xA90

Tasas por 90 minutos en vez de totales, para no mezclar producción con
minutos jugados. Separa perfiles: killers puros (abajo a la derecha),
creadores puros (arriba a la izquierda) y todocampo (arriba a la
derecha) — las líneas punteadas marcan el promedio de cada eje.

In [6]:
scatter_data = temporadas[SEASON]

fig = go.Figure()
for liga, sub in scatter_data:
    fig.add_trace(go.Scatter(
        x=sub["xG90"], y=sub["xA90"], mode="markers", name=liga,
        marker=dict(color=league_color(liga), size=BASE_SIZE, opacity=0.75,
                    line=dict(width=0.5, color="white")),
        customdata=sub[["player", "team"]],
        hovertemplate="<b>%{customdata[0]}</b> (%{customdata[1]})<br>"
                       "xG90: %{x:.2f}<br>xA90: %{y:.2f}<extra></extra>",
    ))

# Las referencias van en el promedio de las 5 temporadas, no en el de la
# mostrada: si se movieran, "estar por encima del promedio" significaría algo
# distinto en cada temporada.
fig.add_hline(y=df["xA90"].mean(), line=dict(color="#c3c2b7", width=1, dash="dot"))
fig.add_vline(x=df["xG90"].mean(), line=dict(color="#c3c2b7", width=1, dash="dot"))

sub_p = (f"Killer puro, creador puro o todocampo — jugadores con >={MIN_MINUTES} min · {{temporada}}")
fig.update_layout(
    title=dict(text="Perfil ofensivo: xG90 vs. xA90", subtitle=dict(text=sub_p.format(temporada=SEASON))),
    xaxis_title="xG90", yaxis_title="xA90",
    xaxis=dict(range=eje(df["xG90"])), yaxis=dict(range=eje(df["xA90"])),
)
render_with_sidebar(fig, scatter_data, "xG90", "xA90", name_col="player",
                    search_label="jugador", width=760, height=580, base_size=BASE_SIZE,
                    season_data=temporadas, custom_cols=["player", "team"],
                    subtitle_template=sub_p,
                    insights={'que_mirar': ins.perfil_que_mirar(),
                              'por_que': ins.perfil_por_que(df),
                              'dinamico': por_liga(ins.perfil)})

## 2. Nivel ofensivo por liga

Las tasas por 90' de todos los jugadores con minutos suficientes, liga por
liga — para comparar el nivel goleador y creativo entre las 5 ligas, no
solo entre jugadores individuales.

In [7]:
sub_box = (f"xG por 90' — jugadores con >={MIN_MINUTES} min · {{temporada}}")
fig, controls, _ = league_box_season_html(df, "xG90", "xG90", SEASONS,
                                        hover_fmt=".2f", name_col="player",
                                        subtitle_template=sub_box)
fig.update_layout(title=dict(text="Nivel goleador esperado por liga",
                             subtitle=dict(text=sub_box.format(temporada=SEASON))),
                  yaxis_title="xG90", xaxis_title=None)
render_select_chart(fig, controls, width=800, height=520,
                    insights={'que_mirar': ins.nivel_que_mirar('gol'),
                              'por_que': ins.nivel_por_que('xG90'),
                              'dinamico': {s: dict(zip(('fija', 'salta'),
                                                        ins.nivel(df, 'xG90', s)))
                                            for s in SEASONS}})

In [8]:
sub_box = (f"xA por 90' — jugadores con >={MIN_MINUTES} min · {{temporada}}")
fig, controls, _ = league_box_season_html(df, "xA90", "xA90", SEASONS,
                                        hover_fmt=".2f", name_col="player",
                                        subtitle_template=sub_box)
fig.update_layout(title=dict(text="Nivel de creación esperado por liga",
                             subtitle=dict(text=sub_box.format(temporada=SEASON))),
                  yaxis_title="xA90", xaxis_title=None)
render_select_chart(fig, controls, width=800, height=520,
                    insights={'que_mirar': ins.nivel_que_mirar('juego'),
                              'por_que': ins.nivel_por_que('xA90'),
                              'dinamico': {s: dict(zip(('fija', 'salta'),
                                                        ins.nivel(df, 'xA90', s)))
                                            for s in SEASONS}})

## 3. Mapa de calor de los tiros

Los gráficos anteriores miran a los jugadores; este mira a los **remates**. Es el
único de la sección que no sale de `players_all_seasons.csv` sino del detalle de
tiros de Understat, y responde a una pregunta de la misma familia: no cuántos
goles hace alguien, sino **desde dónde se dispara**.

Dos filtros: temporada y tipo de tiro. El color de cada celda es el *porcentaje
de los tiros de la selección* que salió de ahí, y no el número de tiros, para
que dos filtros de tamaños muy distintos (164.000 tiros de jugada abierta contra
6.900 de falta directa) se puedan comparar mirando la forma del mapa.

El armado vive en `shot_map.py` y no en este notebook: es el mismo módulo que
usa el sitio, así que el gráfico de acá y el de `jugadores.html` no se pueden
separar. Ahí están documentadas también las dos decisiones de método — por qué
se sacan los penaltis y por qué la izquierda del mapa es la izquierda del
ataque.

In [9]:
import shot_map

tiros, descartes = shot_map.load("../data/processed/shots_all_seasons.csv")
TEMPORADAS_TIROS = sorted(tiros["temporada"].unique())
print(f"{len(tiros)} tiros en el mapa · fuera: {descartes['penales']} penaltis, "
      f"{descartes['fuera']} desde más de {shot_map.PROFUNDIDAD:.0f} m")

shot_map.render_shot_map(tiros, descartes, TEMPORADAS_TIROS)

220720 tiros en el mapa · fuera: 2771 penaltis, 1202 desde más de 36 m
